# Provider Benchmark — Chem Guide AI Tutor

Compare OpenAI, Anthropic (Claude), and Google Gemini on three tasks:
1. **Problem generation** — structured output correctness + latency
2. **Hint generation** — no-leak compliance + latency  
3. **Error classification** — category accuracy + latency

## Setup
```bash
pip install '.[notebooks]'
cp .env.example .env  # fill in all three provider keys
```

In [ ]:
import asyncio
import os
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Add backend root to path so we can import the app
sys.path.insert(0, str(Path().resolve().parent))
load_dotenv(Path().resolve().parent / '.env')

from app.services.ai.provider import ProviderFactory
from app.services.ai.tutor_service import TutorService
from app.domain.schemas.tutor import ProblemOutput, HintOutput, ErrorClassificationOutput

PROVIDERS = ["openai", "anthropic", "gemini"]

# How many runs per task per provider
N_RUNS = 3
print(f"Benchmarking {len(PROVIDERS)} providers × {N_RUNS} runs")

## Task 1 — Problem Generation

**Metrics**
- `latency_s`: wall-clock seconds
- `valid_structure`: did we get exactly 5 steps? (0/1)
- `has_correct_types`: are steps 1-2 `given` and 3-5 `interactive`? (0/1)

In [ ]:
PROBLEM_PARAMS = dict(
    chapter_id="kinetics",
    topic_index=0,
    topic_name="Zero-Order Kinetics",
    difficulty="medium",
)

def score_problem(problem: ProblemOutput) -> dict:
    steps = problem.steps
    valid_count = len(steps) == 5
    correct_types = (
        all(s.type == "given" for s in steps[:2])
        and all(s.type == "interactive" for s in steps[2:])
    )
    has_answers = all(
        s.correct_answer is not None for s in steps if s.type == "interactive"
    )
    return {
        "valid_structure": int(valid_count),
        "correct_step_types": int(correct_types),
        "interactive_steps_have_answers": int(has_answers),
    }

async def benchmark_problem_generation():
    results = []
    for provider_name in PROVIDERS:
        try:
            provider = ProviderFactory.get(provider_name)
        except Exception as e:
            print(f"  [{provider_name}] Skipped — {e}")
            continue
        service = TutorService(provider=provider)
        for run in range(N_RUNS):
            try:
                t0 = time.perf_counter()
                problem = await service.generate_problem(**PROBLEM_PARAMS)
                latency = time.perf_counter() - t0
                scores = score_problem(problem)
                results.append({
                    "provider": provider_name,
                    "run": run + 1,
                    "latency_s": round(latency, 2),
                    "error": None,
                    **scores,
                })
            except Exception as e:
                results.append({"provider": provider_name, "run": run + 1, "error": str(e)})
    return pd.DataFrame(results)

df_problems = await benchmark_problem_generation()
df_problems

## Task 2 — Hint Generation

**Metrics**
- `latency_s`: wall-clock seconds
- `no_answer_leak`: hint does not contain the correct answer string (0/1)

In [ ]:
HINT_PARAMS = dict(
    step_label="Calculate",
    step_instruction="Perform the arithmetic to find the rate constant k.",
    student_input="0.5",
    correct_answer="2.5e-3",
    attempt_count=2,
    problem_context="Zero-order kinetics: [A] = [A]0 - kt",
)

def score_hint(hint: HintOutput, correct_answer: str) -> dict:
    # Check the hint doesn't contain the exact correct value
    no_leak = correct_answer not in hint.hint and "2.5" not in hint.hint
    has_content = len(hint.hint) > 20
    return {
        "no_answer_leak": int(no_leak),
        "hint_non_trivial": int(has_content),
        "hint_level": hint.hint_level,
    }

async def benchmark_hint_generation():
    results = []
    for provider_name in PROVIDERS:
        try:
            provider = ProviderFactory.get(provider_name)
        except Exception as e:
            print(f"  [{provider_name}] Skipped — {e}")
            continue
        service = TutorService(provider=provider)
        for run in range(N_RUNS):
            try:
                t0 = time.perf_counter()
                hint = await service.generate_hint(**HINT_PARAMS)
                latency = time.perf_counter() - t0
                scores = score_hint(hint, HINT_PARAMS["correct_answer"])
                results.append({
                    "provider": provider_name,
                    "run": run + 1,
                    "latency_s": round(latency, 2),
                    "error": None,
                    **scores,
                })
            except Exception as e:
                results.append({"provider": provider_name, "run": run + 1, "error": str(e)})
    return pd.DataFrame(results)

df_hints = await benchmark_hint_generation()
df_hints

## Task 3 — Error Classification

**Metrics**
- `latency_s`: wall-clock seconds
- `valid_categories`: all category values are in the allowed enum (0/1)
- `has_insight`: insight string is non-empty

In [ ]:
CLASSIFY_PARAMS = dict(
    incorrect_steps=[
        {"id": "s3", "label": "Substitute", "studentInput": "k = 0.5 / (2*0.1)", "expectedValue": "k = 0.5", "timeSpent": 45},
    ],
    all_steps=[
        {"id": "s1", "label": "Equation", "isCorrect": True, "timeSpent": 10},
        {"id": "s2", "label": "Knowns", "isCorrect": True, "timeSpent": 8},
        {"id": "s3", "label": "Substitute", "isCorrect": False, "timeSpent": 45},
        {"id": "s4", "label": "Calculate", "isCorrect": False, "timeSpent": 30},
        {"id": "s5", "label": "Answer", "isCorrect": False, "timeSpent": 20},
    ],
    problem_context="Zero-order kinetics: [A] = [A]0 - kt, finding k given [A]=0.3M, [A]0=0.5M, t=80s",
)

VALID_CATEGORIES = {"conceptual", "procedural", "computational", "representation"}

def score_classification(result: ErrorClassificationOutput) -> dict:
    all_valid = all(e.category in VALID_CATEGORIES for e in result.errors)
    return {
        "valid_categories": int(all_valid),
        "error_count": len(result.errors),
        "has_insight": int(len(result.insight) > 20),
    }

async def benchmark_classification():
    results = []
    for provider_name in PROVIDERS:
        try:
            provider = ProviderFactory.get(provider_name)
        except Exception as e:
            print(f"  [{provider_name}] Skipped — {e}")
            continue
        service = TutorService(provider=provider)
        for run in range(N_RUNS):
            try:
                t0 = time.perf_counter()
                result = await service.classify_errors(**CLASSIFY_PARAMS)
                latency = time.perf_counter() - t0
                scores = score_classification(result)
                results.append({
                    "provider": provider_name,
                    "run": run + 1,
                    "latency_s": round(latency, 2),
                    "error": None,
                    **scores,
                })
            except Exception as e:
                results.append({"provider": provider_name, "run": run + 1, "error": str(e)})
    return pd.DataFrame(results)

df_classify = await benchmark_classification()
df_classify

## Summary

In [ ]:
def summarise(df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    numeric = df.select_dtypes(include='number').columns.tolist()
    summary = df.groupby("provider")[numeric].mean().round(3)
    summary.columns = [f"{task_name}__{c}" for c in summary.columns]
    return summary

summary = pd.concat([
    summarise(df_problems, "problem"),
    summarise(df_hints, "hint"),
    summarise(df_classify, "classify"),
], axis=1)

print("\n=== BENCHMARK SUMMARY ===")
print(summary.to_string())

In [ ]:
# ── Latency comparison chart ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
tasks = [
    (df_problems, "Problem Generation"),
    (df_hints, "Hint Generation"),
    (df_classify, "Error Classification"),
]

for ax, (df, title) in zip(axes, tasks):
    if "latency_s" not in df.columns:
        continue
    grouped = df.groupby("provider")["latency_s"].mean()
    grouped.plot.bar(ax=ax, color=["#4C8EDA", "#E67E22", "#2ECC71"], edgecolor="black")
    ax.set_title(title)
    ax.set_ylabel("Avg latency (s)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.suptitle("AI Provider Latency Benchmark", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_latency.png", dpi=150)
plt.show()
print("Saved → benchmark_latency.png")